## Agent
- a simple agentic workflow to get updated information about a given question
    - information collector
    - information presenter
- tools
    - duck duck go search tool from langchain
- api
    - using `dotenv`

In [7]:
# loading api key
import dotenv
dotenv.load_dotenv(dotenv.find_dotenv())

True

## LLM
- llm to gernate the ouput
- chat model from the langchain-deepseek model

In [31]:
# getting the model
import os
from langchain_deepseek import ChatDeepSeek
import pprint

pp = pprint.PrettyPrinter(
    indent=4,
    width=60,
    compact=True,
)

llm = ChatDeepSeek(
    model="deepseek-flash",
    temperature=0,
    max_tokens=None,
    max_retries=2,
    api_key=os.environ.get("DEEPSEEK"),
    extra_body={"thinking": {"type": "disabled"}},
)

# info collector
info_collect_prompt = "what is the best tourist place to visit in Frankfurt, Germany?"

response = llm.invoke(info_collect_prompt)
pp.pprint(response.content)

('Frankfurt often gets overlooked as just a business hub, '
 'but it has a great mix of history, culture, and green '
 'spaces. Here are the best places to visit, depending on '
 "what you're into:\n"
 '\n'
 '**Top Overall Pick: Römerberg & the Old Town '
 '(Altstadt)**\n'
 'This is the historic heart of Frankfurt. The Römerberg '
 'square features the iconic half-timbered houses and the '
 "Römer (city hall), and it's the classic postcard image "
 'of the city. The reconstructed Old Town (Neue Altstadt) '
 'nearby is lovely for strolling, with the Kaiserdom '
 '(Imperial Cathedral) right there too. Best for '
 'first-time visitors and photography.\n'
 '\n'
 '**Best for Culture: Museumsufer (Museum Embankment)**\n'
 'A stretch along the south bank of the Main River lined '
 'with world-class museums — the Städel (art), the German '
 'Film Museum, the Communication Museum, and more. Great '
 'if you want substance, and the riverside promenade is '
 'perfect for a walk.\n'
 '\n'
 '**Best

## Custom tools for LLMs
- custom tools
    - from tools module tool
    - doc string act as the documentation to the llm to initiate the call

```python
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers tougether and return the result."""
    return a*b
```

- can use pydantic schema for arguments
```python
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class FetchUserDataInput(BaseModel):
    user_id: int = Field(description="The unique identifier of the User")
    include_address: bool = Field(default=False, description="Weather to incluce user's physical address")

@tool(args_schema=FetchUserDataInput)
def fetch_user_data(user_id: int, include_address: bool = False) -> str:
    """Fetch profile information for a specific user ID."""
    return f"User detils for ID {user_id}"
```

In [9]:
from langchain_community.tools import DuckDuckGoSearchResults

search_tool = DuckDuckGoSearchResults()
query = "what is the best tourist place to visit in Frankfurt, Germany?"
results = search_tool.run(query)
pp.pprint(results)

('snippet: Things to Do in Frankfurt, Germany: See '
 "Tripadvisor's 336,573 traveler reviews and photos of "
 'Frankfurt tourist attractions. Find what to do today, '
 'this weekend, or in November. We have reviews of the '
 'best places to see in Frankfurt. Visit top-rated & '
 'must-see attractions., title: THE 15 BEST Things to Do '
 'in Frankfurt (2026) - Must-See ..., link: '
 'https://www.tripadvisor.com/Attractions-g187337-Activities-Frankfurt_Hesse.html, '
 'snippet: The Broke Backpacker is supported by you. '
 'Clicking through our links may earn us a small affiliate '
 "commission, and that's what allows us to keep producing "
 'free content 🙂 Learn more. Why it’s awesome:Römerberg is '
 'a great place to kick off your trip to Frankfurt. As the '
 'centre of the Old Town, it’s the perfect spot to get an '
 'idea of how Frankfurt lo... See full list on '
 'thebrokebackpacker.com Why it’s awesome:Along both sides '
 'of the River Main, you’ll find this assortment of '
 'fantas

## ReAct agent in LangChain
- break down the task into simple action items

In [33]:
import requests
from bs4 import BeautifulSoup
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool


@tool
def search_web(query: str) -> str:
    """Search DuckDuckGo for web links and brief descriptions."""
    search_tool = DuckDuckGoSearchResults()
    try:
        return search_tool.run(query)
    except Exception as e:
        return f"Error occurred while searching: {str(e)}"


@tool
def scrape_web(url: str) -> str:
    """Scrape and return raw plain text from a web page URL. Use this when search snippets lack detailed information."""
    try:
        # Ignore tracking ad links if passed accidentally
        if "bing.com/aclick" in url or "ad.doubleclick" in url:
            return "Error: Cannot scrape tracking/ad redirect URLs. Please search directly for source website."
    
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
            )
        }

        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        # Strip scripts and styles
        for script in soup(["script", "style"]):
            script.extract()

        text = soup.get_text(separator=" ", strip=True)
        # Return first 3000 characters to prevent context overload
        return text[:3000]
    except Exception as e:
        return f"Failed to scrape webpage: {str(e)}"


In [34]:
from langchain.agents import create_agent

tools = [search_web, scrape_web]
system_message = "You are a helpful assistant that can search the web and scrape web pages for information."

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = system_message
)

query = "which is the best tourist place to visit in Frankfurt, Germany?"

response = agent.invoke({
    "messages": [
    {"role": "user", "content": query}
    ]
})

pp.pprint(response)


{   'messages': [   HumanMessage(content='which is the best tourist place to visit in Frankfurt, Germany?', additional_kwargs={}, response_metadata={}, id='0f1c8fcc-909d-4e7c-9ee6-2239e516c0e8'),
                    AIMessage(content="I'll search for information about the best tourist places to visit in Frankfurt, Germany.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 363, 'total_tokens': 458, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 363}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '4d6d2f86-8faa-432c-8f63-6d6d4b109682', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1f41b-3626-7555-9617-c40050de7bbc-0', tool_calls=[{'name': 'sea

In [37]:
pp.pprint(response["messages"][-1].content)

('For a first-time visitor, the **best single place to '
 'visit in Frankfurt is Römerberg and the New Old Town '
 '(Neue Altstadt)**. It’s the city’s historic heart, with '
 'the iconic **Römer town hall**, half-timbered houses, '
 'St. Bartholomew’s Cathedral nearby, and lively '
 'cafés—ideal for sightseeing, photos, and getting a feel '
 'for Frankfurt.\n'
 '\n'
 'If you have more time, don’t miss:\n'
 '\n'
 '- **Main Tower** – best panoramic view over the skyline\n'
 '- **Städel Museum** – one of Germany’s finest art '
 'museums\n'
 '- **Museumsufer (Museum Embankment)** – museums along '
 'the river\n'
 '- **Palmengarten** – beautiful botanical garden\n'
 '- **Goethe House** – birthplace of Johann Wolfgang von '
 'Goethe\n'
 '\n'
 'So: **Römerberg/Old Town for the classic Frankfurt '
 'experience; Main Tower for the best view.**')


- Annotated is used to add metadata to the object
- `Annoated[T, metatadata 1, metadata 2]`
- `Annotated[list, operator.add]` we are adding the operator.add as a metadata to the list
    - the Langcahin uses this metadata to decide what to do when a new value comes to the list
    - the reduction operation only happend with in the LangChain

In [35]:
from langchain_core.messages import (
    SystemMessage, 
    HumanMessage, 
    AIMessage, 
    ToolMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

MAX_TOOL_CALLS = 5
MAX_EVAL_CALLS = 3

tool_mappings = {tool.name: tool for tool in tools}

# State defenition for the agentic workflow
class State(TypedDict):
    user_input: str
    history: Annotated[list, add]
    tool_call_iteration: int
    eval_iteration: int
    eval_passed: bool

state = State()

## Research Agent
## to answer questions based on the information available and tools provided.
research_system_message = f"""
You are a helpful assistant. Your duty is to answer user question based on the information you have.
If you don't know the answer, you should say "I don't know". Don't make up answers.
You are allowed to use tools, the following tools are available to you:
{'\n -'.join([f'{tool.name}: {tool.description}' for tool in tools])}
"""

research_prompt = ChatPromptTemplate.from_messages([
    ("system", research_system_message),
    MessagesPlaceholder(variable_name="history"),
    ("user", "{user_input}")
])

def research_agent(state: State) -> AIMessage:
    tool_call_iteration = state.get("tool_call_iteration")
    if tool_call_iteration < MAX_TOOL_CALLS:
        llm_with_tools = llm.bind_tools(tools)
    else:
        llm_with_tools = llm
    chain = research_prompt | llm_with_tools
    response = chain.invoke(state)
    return {"history": [response]}

# tool call node
def tool_call_node(state: State) -> ToolMessage:
    last_response = state.get("history")[-1]
    iterations = state.get("tool_call_iteration", 0)

    iterations += 1
    tool_messages = []
    for tool_call in last_response.tool_calls:
        name = tool_call.get("name")
        args = tool_call.get("args")
        id = tool_call.get("id")

        if name in tool_mappings:
            tool = tool_mappings[name]
            tool_response = tool.invoke(args)

            tool_messages.append(
                ToolMessage(content=tool_response, tool_call_id=id)
            )

    return {
        "history": tool_messages,
        "tool_call_iteration": iterations
    }

# Evaluation node
class EvaluationResult(BaseModel):
    passed: bool = Field(description="True if the answer is accurate, and fully answered the question")
    feedback: str = Field(description = "Feedback given by the user indicateing the shortcomings of the answer provided")

def evaluator(state: State) -> AIMessage:
    last_response = state.get("history")[-1]

    evaluation_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that evaluates the quality of the answer provided by the agent."),
        ("user", f"""
        Please evaluate the answer provided by the agent. 
        user input: {state.get("user_input")}
        agent answer: {last_response.content}
        Is it complete and accurate? If not, please provide feedback.
        """)
    ])
    llm_with_structured_output = llm.with_structured_output(EvaluationResult)
    chain = evaluation_prompt | llm_with_structured_output
    response = chain.invoke(state)
    return {
        "history": [response],
        "tool_call_iteration": 0, # rest the tool call counter
        "eval_iteration": state.get("eval_iteration", 0) + 1, # count the evaluation
        "eval_passed": response.passed
    }

def tool_cal_router(state: State) -> str:
    last_state = state.get("history")[-1]

    if last_state.tool_calls:
        return "tool_call_node"
    elif state.get("eval_iteration") < MAX_EVAL_CALLS or not state.get("eval_passed", False):
        return "evaluator"
    else:
        return END

graph = StateGraph(State)

graph.add_node("research_agent", research_agent)
graph.add_node("tool_call_node", tool_call_node)
graph.add_node("evaluator", evaluator)

graph.add_edge(START, "research_agent")
graph.add_edge("tool_call_node", "research_agent")
graph.add_edge("evaluator", "research_agent")

# conditional edge to tool_call and evaluator nodes
graph.add_conditional_edges(
    "research_agent", 
    tool_cal_router, 
    {
    "tool_call_node": "tool_call_node",
    "evaluator": "evaluator",
    END: END
    }
)  

app = graph.compile()

user_input = "which is the best tourist place to visit in Frankfurt, Germany?"
response = app.invoke({
    "user_input": user_input,
    "history": [],
    "tool_call_iteration": 0,
    "eval_iteration": 0
})
result = response.get("history")[-1]
print(result.content)


NotImplementedError: Unsupported message type: <class '__main__.EvaluationResult'>
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/MESSAGE_COERCION_FAILURE 

In [18]:
f"""
You are a helpful assistent. Your duity is to answer user question based on the information you have.
If you don't know the answer, you should say "I don't know". Don't make up answers.
You are allowed to use tools, the following tools are available to you:
{"\n -".join([f'{tool.name}: {tool.description}' for tool in tools])}
"""

'\nYou are a helpful assistent. Your duity is to answer user question based on the information you have.\nIf you don\'t know the answer, you should say "I don\'t know". Don\'t make up answers.\nYou are allowed to use tools, the following tools are available to you:\nsearch_web: Search DuckDuckGo for web links and brief descriptions.\n -scrape_web: Scrape and return raw plain text from a web page URL. Use this when search snippets lack detailed information.\n'